# Train, Validate, and Reward-Shape with the Time2Success Model

Three stages in one notebook:
1. Train the MLP regressor on `state_space_time2success_dataset/dataset.npz`
2. Validate qualitatively with a successful-episode video and a failed-episode video
3. Use the trained model as a shaped reward to train a new policy, compare against stage 1

Run and check each stage before moving to the next — don't run the whole
notebook top-to-bottom blindly, since stage 3 is a multi-hour commitment.

In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
import json, os
import gymnasium as gym
import metaworld
import imageio
from stable_baselines3 import SAC
import collections

# Model definition. NOTE: train_loader/val_loader come from the
# NORMALIZATION cell above (X_train_n/y_train_n) — this cell does NOT
# rebuild them from raw data, unlike the earlier version of this notebook
# which silently overwrote the normalized loaders here (that was Bug 1).
DATASET_PATH = "./data/state_space_dataset/dataset.npz"
data = np.load(DATASET_PATH)
X, y_steps, y_seconds, episode_ids = data["X"], data["y_steps"], data["y_seconds"], data["episode_ids"]

with open("./data/state_space_dataset/dataset_summary.json") as f:
    summary = json.load(f)
print(summary)

OBS_DIM = X.shape[1]

class Time2SuccessModel(nn.Module):
    def __init__(self, obs_dim, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

device = "cuda" if torch.cuda.is_available() else "cpu"
t2s_model = Time2SuccessModel(obs_dim=OBS_DIM).to(device)
optimizer = torch.optim.Adam(t2s_model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()


TASK_NAME = "peg-insert-side-v3"
SUCCESS_KEY = "success"

def make_env(seed=0, render_mode=None):
    return gym.make("Meta-World/MT1", env_name=TASK_NAME, seed=seed, render_mode=render_mode)

_probe = make_env(seed=0)
DT = getattr(_probe.unwrapped, "dt", _probe.unwrapped.model.opt.timestep)
OBS_DIM = _probe.observation_space.shape[0]
_probe.close()

{'total_rows': 5771, 'total_episodes': 100, 'obs_dim': 39, 'dt_sec': 0.0125, 'source_policy': './checkpoints/peg_insert_side/sac_peg_insert_final', 'successful_only': True, 'checkpoint_diversity': False}


d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:34: UserWarning: WARN: A Box observation space maximum and minimum values are equal.
  logger.warn("A Box observation space maximum and minimum values are equal.")


In [8]:
import collections
import gymnasium as gym
import json
import numpy as np
import torch
import torch.nn as nn

# ============================================================
# 1. Load Normalization & Frozen Time2Success Model
# ============================================================
norm = np.load("./checkpoints/time2success/normalization.npz")
SHAPING_X_MEAN, SHAPING_X_STD = norm["X_mean"], norm["X_std"]
SHAPING_Y_MEAN, SHAPING_Y_STD = norm["y_mean"].item(), norm["y_std"].item()

shaping_t2s_model = Time2SuccessModel(obs_dim=OBS_DIM).to(device)
shaping_t2s_model.load_state_dict(
    torch.load("./checkpoints/time2success/time2success_state_model_best.pt")
)
shaping_t2s_model.eval()  # Strictly frozen for RL training
# ============================================================

GAMMA = 0.99


class PureTime2SuccessRewardWrapper(gym.Wrapper):

  def __init__(
      self,
      env,
      t2s_model,
      device,
      x_mean,
      x_std,
      y_mean,
      y_std,
      gamma=GAMMA,
      shaping_scale=1.0,
      success_bonus=50.0,
      stall_window=20,
  ):
    super().__init__(env)
    self.t2s_model = t2s_model
    self.device = device
    self.x_mean, self.x_std = x_mean, x_std
    self.y_mean, self.y_std = y_mean, y_std
    self.gamma = gamma
    self.shaping_scale = shaping_scale
    self.success_bonus = (
        success_bonus  # Ground-truth terminal bonus upon success
    )
    self.stall_window = stall_window  # Fixed missing attribute

    self._last_phi = 0.0
    self._recent_predictions = collections.deque(maxlen=self.stall_window)

  @torch.no_grad()
  def _potential(self, obs):
    # Normalized forward pass through the frozen T2S regressor
    x_norm = (obs - self.x_mean) / self.x_std
    x = torch.tensor(x_norm, dtype=torch.float32).unsqueeze(0).to(self.device)
    pred_norm = self.t2s_model(x).item()
    pred_steps = pred_norm * self.y_std + self.y_mean  # De-normalize
    return -pred_steps  # Closer to success (lower remaining steps) = higher potential

  def reset(self, **kwargs):
    obs, info = self.env.reset(**kwargs)
    self._last_phi = self._potential(obs)

    self._recent_predictions.clear()
    self._recent_predictions.append(-self._last_phi)  # Track initial t2s
    return obs, info

  def step(self, action):
      # Step environment — Meta-World's internal reward is discarded
      obs, _base_reward, terminated, truncated, info = self.env.step(action)

      phi_next = self._potential(obs)
      predicted_now = -phi_next

      # Stall detection: cut rollout short if policy fails to progress over stall_window
      if (
          len(self._recent_predictions) == self.stall_window
          and predicted_now >= max(self._recent_predictions)
      ):
        truncated = True

      self._recent_predictions.append(predicted_now)

      # Pure Potential-Based Time Difference Signal
      pure_t2s_reward = self.gamma * phi_next - self._last_phi
      self._last_phi = phi_next

      # Scale the pure signal
      total_reward = self.shaping_scale * pure_t2s_reward

      return obs, total_reward, terminated, truncated, info


def make_pure_t2s_env(seed=0):
  base_env = make_env(seed=seed)
  return PureTime2SuccessRewardWrapper(
      base_env,
      shaping_t2s_model,
      device,
      SHAPING_X_MEAN,
      SHAPING_X_STD,
      SHAPING_Y_MEAN,
      SHAPING_Y_STD,
      gamma=GAMMA,
      shaping_scale=1.0,
      success_bonus=50.0,
      stall_window=20,
  )

In [9]:
import time
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor

N_ENVS = 6
pure_train_env = DummyVecEnv(
    [lambda i=i: make_pure_t2s_env(seed=i) for i in range(N_ENVS)]
)
pure_train_env = VecMonitor(pure_train_env)

# Evaluation is kept on raw Meta-World environment to evaluate TRUE success rate
pure_eval_env = make_env(seed=1000)

pure_model = SAC(
    policy="MlpPolicy",
    env=pure_train_env,
    learning_rate=3e-4,
    buffer_size=1_000_000,
    batch_size=256,
    tau=0.005,
    gamma=GAMMA,
    ent_coef="auto",
    policy_kwargs=dict(net_arch=[400, 400]),
    tensorboard_log="./tb_logs/peg_insert_side_pure_t2s",
    verbose=1,
    seed=0,
)

Using cpu device


Reuse the same eval/checkpoint callback pattern from stage 1, pointed at a separate directory.

In [10]:
class SuccessCallback(BaseCallback):
    def __init__(self, eval_env, eval_freq=10_000, n_eval_episodes=10,
                 ckpt_dir="./checkpoints/peg_insert_side_shaped", ckpt_freq=100_000, verbose=1):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes
        self.ckpt_dir = ckpt_dir
        self.ckpt_freq = ckpt_freq
        os.makedirs(ckpt_dir, exist_ok=True)
        self.history = []

    def _run_eval_episode(self):
        obs, _ = self.eval_env.reset()
        success_step = None
        for t in range(500):
            action, _ = self.model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = self.eval_env.step(action)
            if info.get(SUCCESS_KEY, 0) and success_step is None:
                success_step = t
            if terminated or truncated:
                break
        return success_step

    def _on_step(self) -> bool:
        if self.num_timesteps % self.ckpt_freq < self.training_env.num_envs:
            self.model.save(os.path.join(self.ckpt_dir, f"shaped_{self.num_timesteps}.zip"))

        if self.num_timesteps % self.eval_freq < self.training_env.num_envs:
            steps = [self._run_eval_episode() for _ in range(self.n_eval_episodes)]
            success_rate = sum(s is not None for s in steps) / self.n_eval_episodes
            times = [s for s in steps if s is not None]
            mean_t2s = float(np.mean(times)) if times else None
            self.logger.record("eval/success_rate", success_rate)
            if mean_t2s is not None:
                self.logger.record("eval/mean_time_to_success", mean_t2s)
            self.history.append(dict(step=self.num_timesteps, success_rate=success_rate,
                                       mean_time_to_success=mean_t2s, timestamp=time.time()))
            with open(os.path.join(self.ckpt_dir, "eval_history.json"), "w") as f:
                json.dump(self.history, f, indent=2)
            if self.verbose:
                print(f"[eval @ {self.num_timesteps}] success_rate={success_rate:.2f} mean_t2s={mean_t2s}")
        return True

shaped_callback = SuccessCallback(eval_env=shaped_eval_env)

NameError: name 'shaped_eval_env' is not defined

**Smoke test — run this first, same pattern as stage 1:**

In [ ]:
shaped_model.learn(total_timesteps=5_000, callback=shaped_callback, tb_log_name="shaped_smoke")
print("Smoke test complete — check eval_history.json wrote correctly before scaling up")

**Full run** — only after the smoke test looks right. Same step budget
as stage 1 for a fair comparison. This is a multi-hour commitment on a
laptop; use the `.py` script pattern discussed earlier for long runs, or set
this cell running and check back rather than watching it live.

In [ ]:
TOTAL_TIMESTEPS = 3_000_000

shaped_model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=shaped_callback,
    tb_log_name="shaped_run1",
    progress_bar=True,
    reset_num_timesteps=False,
)
shaped_model.save("./checkpoints/peg_insert_side_shaped/sac_peg_insert_shaped_final")
shaped_train_env.close()
shaped_eval_env.close()
print("Shaped training complete")

## Comparing against stage 1

Once both `eval_history.json` files exist (original at
`./checkpoints/peg_insert_side/eval_history.json`, shaped at
`./checkpoints/peg_insert_side_shaped/eval_history.json`), plot success rate
vs. step for both on the same axes — this is the actual answer to
"did this help," not any number from stage 1 or 2 alone.

In [ ]:
with open("./checkpoints/peg_insert_side/eval_history.json") as f:
    orig_history = json.load(f)
with open("./checkpoints/peg_insert_side_shaped/eval_history.json") as f:
    shaped_history = json.load(f)

orig_df = pd.DataFrame(orig_history) if 'pd' in dir() else __import__('pandas').DataFrame(orig_history)
shaped_df = __import__('pandas').DataFrame(shaped_history)

plt.figure(figsize=(7,4))
plt.plot(orig_df["step"], orig_df["success_rate"], label="original (dense reward only)")
plt.plot(shaped_df["step"], shaped_df["success_rate"], label="shaped (+ time2success)")
plt.xlabel("step")
plt.ylabel("success rate")
plt.legend()
plt.title("Original vs. shaped-reward training")
plt.show()

## Pure time-difference reward, with a ground-truth success bonus

Replaces `shaped_reward = reward + shaping` with `shaped_reward = shaping`
(the time-difference alone), **plus** a large constant bonus only when the
simulator's real `success` flag fires — that bonus is ground truth, not
model-derived, and exists specifically so the policy has at least one
unambiguous signal the time2success model can't get wrong.

In [ ]:
class Time2SuccessDiffRewardWrapper(gym.Wrapper):
    def __init__(self, env, t2s_model, device, x_mean, x_std, y_mean, y_std,
                 gamma=0.99, shaping_scale=1.0, success_bonus=50.0):
        super().__init__(env)
        self.t2s_model = t2s_model
        self.device = device
        self.x_mean, self.x_std = x_mean, x_std
        self.y_mean, self.y_std = y_mean, y_std
        self.gamma = gamma
        self.shaping_scale = shaping_scale
        self.success_bonus = success_bonus  # ground-truth signal, not model-derived
        self._last_obs = None

    @torch.no_grad()
    def _potential(self, obs):
        x_norm = (obs - self.x_mean) / self.x_std
        x = torch.tensor(x_norm, dtype=torch.float32).unsqueeze(0).to(self.device)
        pred_norm = self.t2s_model(x).item()
        pred_steps = pred_norm * self.y_std + self.y_mean
        return -pred_steps

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._last_obs = obs
        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        phi_s = self._potential(self._last_obs)
        phi_s_next = self._potential(obs)
        time_diff_reward = self.gamma * phi_s_next - phi_s  # pure model-derived signal

        # Ground-truth bonus: unaffected by anything the model gets wrong
        gt_bonus = self.success_bonus if info.get(SUCCESS_KEY, 0) else 0.0

        shaped_reward = self.shaping_scale * time_diff_reward + gt_bonus
        self._last_obs = obs
        return obs, shaped_reward, terminated, truncated, info

def make_diff_env(seed=0):
    base_env = make_env(seed=seed)
    return Time2SuccessDiffRewardWrapper(
        base_env, shaping_t2s_model, device,
        SHAPING_X_MEAN, SHAPING_X_STD, SHAPING_Y_MEAN, SHAPING_Y_STD,
    )

### Smoke test — pure time-difference + ground-truth success bonus

Same 5k-step pattern as every other smoke test in this project. Watch for:
- `shaped_reward` values that aren't wildly huge or tiny (sanity on `shaping_scale`)
- `eval/success_rate` in the printed callback output actually being non-nonsensical
  (it's computed on the REAL env/success flag regardless of what reward the
  policy trained on, so this is your real signal even during the smoke test)

In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor

N_ENVS = 6
diff_train_env = DummyVecEnv([lambda i=i: make_diff_env(seed=i) for i in range(N_ENVS)])
diff_train_env = VecMonitor(diff_train_env)
diff_eval_env = make_env(seed=1000)  # eval always on RAW env/success, never shaped

diff_model = SAC(
    policy="MlpPolicy",
    env=diff_train_env,
    learning_rate=3e-4,
    buffer_size=1_000_000,
    batch_size=256,
    tau=0.005,
    gamma=0.99,
    ent_coef="auto",
    policy_kwargs=dict(net_arch=[400, 400]),
    tensorboard_log="./tb_logs/peg_insert_side_timediff",  # separate from both earlier runs
    verbose=1,
    seed=0,
)

diff_callback = SuccessCallback(
    eval_env=diff_eval_env,
    ckpt_dir="./checkpoints/peg_insert_side_timediff",
)

# --- Smoke test: 5k steps, confirm the loop runs and eval_history.json writes ---
diff_model.learn(total_timesteps=5_000, callback=diff_callback, tb_log_name="timediff_smoke")
print("Smoke test complete. Check eval_history.json under ./checkpoints/peg_insert_side_timediff/")

**Before scaling to the full run**, worth a quick manual check on reward
magnitude — print a few raw `time_diff_reward` and `gt_bonus` values from
one rollout, and compare their scales. If `time_diff_reward` per step is,
say, ~0.01 and `success_bonus=50` swamps it entirely (or vice versa), that's
a `shaping_scale`/`success_bonus` tuning problem worth fixing before the long
run, not after.